# NCT-CRC / CRC-VAL Training - Safe Memory Version

This Kaggle notebook trains EfficientNet-B0 on `NCT-CRC-HE-100K` and evaluates on `CRC-VAL-HE-7K`.

It is designed to avoid the RAM crash you hit by using:

- `torchvision.datasets.ImageFolder`, which reads images from disk instead of loading all images into RAM
- conservative `BATCH_SIZE = 64`
- conservative `NUM_WORKERS = 2`
- `persistent_workers=False`
- `prefetch_factor=1`
- a smoke test before full training
- progress bars during training and validation
- automatic model saving after each epoch

## 1. Before running

In Kaggle:

1. Set accelerator to **T4 x2** if available.
2. Add both inputs as **Datasets**, not notebooks:
   - `NCT-CRC-HE-100K`
   - `CRC-VAL-HE-7K`
3. Do not download Zenodo ZIP files into `/kaggle/working`.
4. Restart the session after changing inputs or accelerator.

In [ ]:
import os
import time
import shutil
import random
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset

import torchvision
from torchvision import transforms
from torchvision.datasets import ImageFolder
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

from tqdm.auto import tqdm
import matplotlib.pyplot as plt

try:
    from sklearn.model_selection import train_test_split
    from sklearn.metrics import classification_report, confusion_matrix
    SKLEARN_AVAILABLE = True
except Exception as e:
    print("sklearn import failed; fallback split will be used:", e)
    SKLEARN_AVAILABLE = False

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.benchmark = True

OUTPUT_DIR = Path("/kaggle/working")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Torch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(i, torch.cuda.get_device_name(i))


## 2. Safe settings

These are intentionally conservative. After a successful run, you can try `BATCH_SIZE = 96`, but avoid `BATCH_SIZE = 128/256` with many workers on Kaggle because it can crash CPU RAM.

In [ ]:
IMG_SIZE = 224
BATCH_SIZE = 64
NUM_WORKERS = 2
EPOCHS = 5
VAL_SIZE = 0.10

LEARNING_RATE = 3e-4
WEIGHT_DECAY = 1e-4

USE_SMOKE_TEST = True
SMOKE_BATCHES = 4

LABEL_NAMES = ["ADI", "BACK", "DEB", "LYM", "MUC", "MUS", "NORM", "STR", "TUM"]

print("IMG_SIZE:", IMG_SIZE)
print("BATCH_SIZE:", BATCH_SIZE)
print("NUM_WORKERS:", NUM_WORKERS)
print("EPOCHS:", EPOCHS)


## 3. Optional cleanup

Only run this cell if `/kaggle/working` is full from previous failed downloads or training attempts.

In [ ]:
def cleanup_working_crc_files():
    root = Path("/kaggle/working")
    patterns = ["*NCT*", "*CRC*", "*VAL*", "*.zip", "*.pt", "*.pth", "*.csv", "*.png"]

    print("Before cleanup:")
    os.system("du -sh /kaggle/working || true")

    for pattern in patterns:
        for p in root.glob(pattern):
            try:
                if p.is_file() or p.is_symlink():
                    p.unlink()
                    print("Deleted file:", p)
                elif p.is_dir():
                    shutil.rmtree(p)
                    print("Deleted folder:", p)
            except Exception as e:
                print("Could not delete:", p, e)

    print("After cleanup:")
    os.system("du -sh /kaggle/working || true")

# Uncomment the next line only if /kaggle/working is full:
# cleanup_working_crc_files()


## 4. Find dataset folders under `/kaggle/input`

This searches recursively, so it works even when Kaggle mounts your inputs under `/kaggle/input/datasets/...`.

In [ ]:
def looks_like_crc_folder(path: Path) -> bool:
    return path.is_dir() and all((path / label).is_dir() for label in LABEL_NAMES)

def count_images_fast(folder: Path):
    exts = {".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp"}
    total = 0
    by_class = {}

    for label in LABEL_NAMES:
        class_dir = folder / label
        count = sum(1 for f in class_dir.iterdir() if f.suffix.lower() in exts)
        by_class[label] = count
        total += count

    return total, by_class

def find_crc_folders(input_root="/kaggle/input"):
    input_root = Path(input_root)
    candidates = []

    print("Top-level /kaggle/input contents:")
    for p in input_root.iterdir():
        print(" ", p)

    for name in ["NCT-CRC-HE-100K", "CRC-VAL-HE-7K"]:
        for p in input_root.glob(f"**/{name}"):
            if looks_like_crc_folder(p):
                candidates.append(p)

    for p in input_root.glob("**/*"):
        if looks_like_crc_folder(p) and p not in candidates:
            candidates.append(p)

    unique = []
    seen = set()
    for p in candidates:
        s = str(p.resolve())
        if s not in seen:
            unique.append(p)
            seen.add(s)

    return unique

crc_folders = find_crc_folders("/kaggle/input")

print()
print("Found CRC-like folders:")
folder_info = []
for p in crc_folders:
    total, by_class = count_images_fast(p)
    folder_info.append((p, total, by_class))
    print(f" {p} -> {total:,} images")
    print("  ", by_class)

TRAIN_DIR = None
TEST_DIR = None

for p, total, by_class in folder_info:
    text = str(p).upper()
    if "NCT-CRC-HE-100K" in text or "100K" in text:
        TRAIN_DIR = p
    if "CRC-VAL-HE-7K" in text or "VAL" in text or "7K" in text:
        TEST_DIR = p

if (TRAIN_DIR is None or TEST_DIR is None) and len(folder_info) >= 2:
    sorted_info = sorted(folder_info, key=lambda x: x[1], reverse=True)
    TRAIN_DIR = TRAIN_DIR or sorted_info[0][0]
    TEST_DIR = TEST_DIR or sorted_info[1][0]

print()
print("TRAIN_DIR:", TRAIN_DIR)
print("TEST_DIR:", TEST_DIR)

if TRAIN_DIR is None:
    raise FileNotFoundError("Could not find NCT-CRC-HE-100K under /kaggle/input.")
if TEST_DIR is None:
    raise FileNotFoundError("Could not find CRC-VAL-HE-7K under /kaggle/input.")


## 5. Create datasets with `ImageFolder`

This avoids loading the full image dataset into RAM.

In [ ]:
weights = EfficientNet_B0_Weights.IMAGENET1K_V1
mean = weights.transforms().mean
std = weights.transforms().std

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.2),
    transforms.RandomRotation(degrees=10),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std),
])

train_full_for_train = ImageFolder(str(TRAIN_DIR), transform=train_transform)
train_full_for_val = ImageFolder(str(TRAIN_DIR), transform=eval_transform)
test_dataset = ImageFolder(str(TEST_DIR), transform=eval_transform)

print("Train classes:", train_full_for_train.classes)
print("Test classes:", test_dataset.classes)
print("Train total:", len(train_full_for_train))
print("Test total:", len(test_dataset))

if train_full_for_train.classes != test_dataset.classes:
    raise ValueError("Train and test class orders do not match.")


## 6. Train/validation split

The 100K dataset is split into train/validation. The 7K dataset is kept for final testing.

In [ ]:
targets = np.array(train_full_for_train.targets)
indices = np.arange(len(targets))

if SKLEARN_AVAILABLE:
    train_idx, val_idx = train_test_split(
        indices,
        test_size=VAL_SIZE,
        random_state=SEED,
        stratify=targets,
    )
else:
    rng = np.random.default_rng(SEED)
    rng.shuffle(indices)
    val_count = int(len(indices) * VAL_SIZE)
    val_idx = indices[:val_count]
    train_idx = indices[val_count:]

train_dataset = Subset(train_full_for_train, train_idx.tolist())
val_dataset = Subset(train_full_for_val, val_idx.tolist())

print("Train subset:", len(train_dataset))
print("Validation subset:", len(val_dataset))
print("Test dataset:", len(test_dataset))
print("Train class counts:", Counter(targets[train_idx]))
print("Val class counts:", Counter(targets[val_idx]))


## 7. DataLoaders

Safe settings to avoid memory restarts.

In [ ]:
def make_loader(dataset, shuffle=False, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS):
    kwargs = {
        "dataset": dataset,
        "batch_size": batch_size,
        "shuffle": shuffle,
        "num_workers": num_workers,
        "pin_memory": torch.cuda.is_available(),
        "drop_last": False,
    }

    if num_workers > 0:
        kwargs["persistent_workers"] = False
        kwargs["prefetch_factor"] = 1

    return DataLoader(**kwargs)

train_loader = make_loader(train_dataset, shuffle=True)
val_loader = make_loader(val_dataset, shuffle=False)
test_loader = make_loader(test_dataset, shuffle=False)

print("Train batches:", len(train_loader))
print("Val batches:", len(val_loader))
print("Test batches:", len(test_loader))


## 8. Build model

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
use_amp = torch.cuda.is_available()
amp_device = "cuda" if torch.cuda.is_available() else "cpu"

def build_model(num_classes):
    model = efficientnet_b0(weights=weights)
    in_features = model.classifier[1].in_features
    model.classifier[1] = nn.Linear(in_features, num_classes)
    return model

def build_training_objects():
    model = build_model(num_classes=len(train_full_for_train.classes))
    model = model.to(device)

    if torch.cuda.device_count() > 1:
        model = nn.DataParallel(model)
        print(f"Using DataParallel on {torch.cuda.device_count()} GPUs.")
    else:
        print("Using single device:", device)

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

    return model, criterion, optimizer, scheduler, scaler

model, criterion, optimizer, scheduler, scaler = build_training_objects()
print("Model ready.")


## 9. Training/evaluation functions with progress bars

In [ ]:
def accuracy_from_logits(logits, labels):
    preds = logits.argmax(dim=1)
    return (preds == labels).sum().item(), labels.size(0)

def train_one_epoch(model, loader, criterion, optimizer, scaler, device, epoch=None):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    start = time.time()

    pbar = tqdm(loader, desc=f"Train epoch {epoch}", leave=True)

    for images, labels in pbar:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast(amp_device, enabled=use_amp):
            logits = model(images)
            loss = criterion(logits, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        batch_size = labels.size(0)
        running_loss += loss.item() * batch_size

        c, n = accuracy_from_logits(logits.detach(), labels)
        correct += c
        total += n

        pbar.set_postfix({
            "loss": f"{running_loss / total:.4f}",
            "acc": f"{correct / total:.4f}",
        })

    return running_loss / total, correct / total, time.time() - start

@torch.no_grad()
def evaluate(model, loader, criterion, device, epoch=None, name="Val"):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    pbar = tqdm(loader, desc=f"{name} epoch {epoch}", leave=True)

    for images, labels in pbar:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        with torch.amp.autocast(amp_device, enabled=use_amp):
            logits = model(images)
            loss = criterion(logits, labels)

        batch_size = labels.size(0)
        running_loss += loss.item() * batch_size

        c, n = accuracy_from_logits(logits, labels)
        correct += c
        total += n

        pbar.set_postfix({
            "loss": f"{running_loss / total:.4f}",
            "acc": f"{correct / total:.4f}",
        })

    return running_loss / total, correct / total


## 10. Smoke test

Runs a few batches, then rebuilds the model so full training starts clean.

In [ ]:
if USE_SMOKE_TEST:
    smoke_samples = min(BATCH_SIZE * SMOKE_BATCHES, len(train_dataset), len(val_dataset))

    small_train_dataset = Subset(train_dataset, range(smoke_samples))
    small_val_dataset = Subset(val_dataset, range(smoke_samples))

    small_train_loader = make_loader(small_train_dataset, shuffle=True, batch_size=BATCH_SIZE, num_workers=0)
    small_val_loader = make_loader(small_val_dataset, shuffle=False, batch_size=BATCH_SIZE, num_workers=0)

    print("Smoke train samples:", len(small_train_dataset))
    print("Smoke val samples:", len(small_val_dataset))

    smoke_train_loss, smoke_train_acc, smoke_seconds = train_one_epoch(
        model, small_train_loader, criterion, optimizer, scaler, device, epoch="SMOKE"
    )

    smoke_val_loss, smoke_val_acc = evaluate(
        model, small_val_loader, criterion, device, epoch="SMOKE", name="Val-smoke"
    )

    print(
        f"Smoke test complete | "
        f"train loss {smoke_train_loss:.4f} acc {smoke_train_acc:.4f} | "
        f"val loss {smoke_val_loss:.4f} acc {smoke_val_acc:.4f} | "
        f"time {smoke_seconds:.1f}s"
    )

    print("Rebuilding model, optimizer, scheduler, and scaler for a clean full training run...")
    model, criterion, optimizer, scheduler, scaler = build_training_objects()
    print("Clean model ready.")


## 11. Full training

Saves best, last checkpoint, final model, and history CSV.

In [ ]:
history = []
best_val_acc = -1.0

best_model_path = OUTPUT_DIR / "best_efficientnet_b0_nct_crc.pt"
last_checkpoint_path = OUTPUT_DIR / "last_checkpoint_efficientnet_b0_nct_crc.pt"
final_model_path = OUTPUT_DIR / "final_efficientnet_b0_nct_crc.pt"

for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc, seconds = train_one_epoch(
        model, train_loader, criterion, optimizer, scaler, device, epoch=epoch
    )

    val_loss, val_acc = evaluate(
        model, val_loader, criterion, device, epoch=epoch, name="Val"
    )

    scheduler.step()

    row = {
        "epoch": epoch,
        "train_loss": train_loss,
        "train_acc": train_acc,
        "val_loss": val_loss,
        "val_acc": val_acc,
        "seconds": seconds,
        "lr": scheduler.get_last_lr()[0],
    }
    history.append(row)

    print(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"train loss {train_loss:.4f} acc {train_acc:.4f} | "
        f"val loss {val_loss:.4f} acc {val_acc:.4f} | "
        f"time {seconds:.1f}s",
        flush=True,
    )

    state_dict = model.module.state_dict() if isinstance(model, nn.DataParallel) else model.state_dict()

    checkpoint = {
        "model_name": "efficientnet_b0",
        "state_dict": state_dict,
        "classes": train_full_for_train.classes,
        "img_size": IMG_SIZE,
        "epoch": epoch,
        "val_acc": val_acc,
        "history": history,
    }

    torch.save(checkpoint, last_checkpoint_path)

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(checkpoint, best_model_path)
        print("  Saved new best model:", best_model_path, flush=True)

history_df = pd.DataFrame(history)
history_csv = OUTPUT_DIR / "training_history.csv"
history_df.to_csv(history_csv, index=False)

state_dict = model.module.state_dict() if isinstance(model, nn.DataParallel) else model.state_dict()
torch.save({
    "model_name": "efficientnet_b0",
    "state_dict": state_dict,
    "classes": train_full_for_train.classes,
    "img_size": IMG_SIZE,
    "epoch": EPOCHS,
    "history": history,
}, final_model_path)

print("Saved history:", history_csv)
print("Saved final model:", final_model_path)
display(history_df)


## 12. Plot training history

In [ ]:
history_df = pd.read_csv(OUTPUT_DIR / "training_history.csv")

plt.figure(figsize=(8, 5))
plt.plot(history_df["epoch"], history_df["train_loss"], marker="o", label="train_loss")
plt.plot(history_df["epoch"], history_df["val_loss"], marker="o", label="val_loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training and Validation Loss")
plt.legend()
plt.grid(True)
plt.savefig(OUTPUT_DIR / "loss_curve.png", dpi=150, bbox_inches="tight")
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(history_df["epoch"], history_df["train_acc"], marker="o", label="train_acc")
plt.plot(history_df["epoch"], history_df["val_acc"], marker="o", label="val_acc")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Training and Validation Accuracy")
plt.legend()
plt.grid(True)
plt.savefig(OUTPUT_DIR / "accuracy_curve.png", dpi=150, bbox_inches="tight")
plt.show()


## 13. Load best model and evaluate on `CRC-VAL-HE-7K`

In [ ]:
best_checkpoint = torch.load(best_model_path, map_location=device)

test_model = build_model(num_classes=len(best_checkpoint["classes"]))
test_model.load_state_dict(best_checkpoint["state_dict"])
test_model = test_model.to(device)

if torch.cuda.device_count() > 1:
    test_model = nn.DataParallel(test_model)

test_model.eval()

print("Loaded best model from epoch:", best_checkpoint["epoch"])
print("Best validation accuracy:", best_checkpoint["val_acc"])

test_loss, test_acc = evaluate(test_model, test_loader, criterion, device, epoch="BEST", name="Test")

print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_acc:.4f}")


## 14. Confusion matrix and classification report

In [ ]:
@torch.no_grad()
def collect_predictions(model, loader, device):
    model.eval()
    all_preds = []
    all_labels = []

    for images, labels in tqdm(loader, desc="Collect predictions"):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        with torch.amp.autocast(amp_device, enabled=use_amp):
            logits = model(images)

        preds = logits.argmax(dim=1)
        all_preds.extend(preds.cpu().numpy().tolist())
        all_labels.extend(labels.cpu().numpy().tolist())

    return np.array(all_labels), np.array(all_preds)

y_true, y_pred = collect_predictions(test_model, test_loader, device)

predictions_df = pd.DataFrame({
    "true_label_id": y_true,
    "pred_label_id": y_pred,
    "true_label": [test_dataset.classes[i] for i in y_true],
    "pred_label": [test_dataset.classes[i] for i in y_pred],
})

predictions_csv = OUTPUT_DIR / "test_predictions.csv"
predictions_df.to_csv(predictions_csv, index=False)
print("Saved predictions:", predictions_csv)

if SKLEARN_AVAILABLE:
    print(classification_report(y_true, y_pred, target_names=test_dataset.classes, digits=4))

    cm = confusion_matrix(y_true, y_pred)

    plt.figure(figsize=(8, 7))
    plt.imshow(cm)
    plt.title("Confusion Matrix")
    plt.colorbar()
    tick_marks = np.arange(len(test_dataset.classes))
    plt.xticks(tick_marks, test_dataset.classes, rotation=45)
    plt.yticks(tick_marks, test_dataset.classes)
    plt.xlabel("Predicted")
    plt.ylabel("True")

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(j, i, str(cm[i, j]), ha="center", va="center")

    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "confusion_matrix.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("sklearn unavailable; skipping classification report/confusion matrix.")


## 15. Output files

Download these from Kaggle's right sidebar under **Output**:

- `best_efficientnet_b0_nct_crc.pt`
- `last_checkpoint_efficientnet_b0_nct_crc.pt`
- `final_efficientnet_b0_nct_crc.pt`
- `training_history.csv`
- `test_predictions.csv`
- `loss_curve.png`
- `accuracy_curve.png`
- `confusion_matrix.png`

In [ ]:
print("Files in /kaggle/working:")
for p in sorted(Path("/kaggle/working").glob("*")):
    size_mb = p.stat().st_size / (1024 * 1024) if p.is_file() else 0
    print(p, "-", f"{size_mb:.2f} MB")
